# Load hadgems data from newer zarr version

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import sys
import os

from dask import delayed
from IPython.display import display, HTML





from tempfile import TemporaryDirectory
from getpass import getuser
from pathlib import Path
import dask
from dask.distributed import Client, LocalCluster
import bokeh
import subprocess
import re

import warnings
#warnings.filterwarnings('ignore')
scratch_dir = Path('/scratch') / getuser()[0] / getuser()

In [3]:
import xarray
xarray.__version__

'2025.12.0'

In [2]:
from numcodecs import Blosc
# instead of:
# from zarr import Blosc

In [20]:
filename = Path('/scratch') / getuser()[0] / getuser() / 'mhws' / 'OSTIA_pre_1982_2014_FixedDetrend_hob_oct25.zarr'
ds =xr.open_zarr(filename)
ds

<xarray.Dataset> Size: 64GB
Dimensions:         (time: 12052, lat: 720, lon: 1440, dayofyear: 366)
Coordinates:
  * time            (time) datetime64[ns] 96kB 1982-01-01T12:00:00 ... 2014-1...
  * lat             (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * lon             (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
Data variables:
    dat_anomaly     (time, lat, lon) float32 50GB dask.array<chunksize=(25, 720, 1440), meta=np.ndarray>
    extreme_events  (time, lat, lon) bool 12GB dask.array<chunksize=(25, 720, 1440), meta=np.ndarray>
    mask            (lat, lon) bool 1MB dask.array<chunksize=(360, 720), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 2GB dask.array<chunksize=(90, 180, 46), meta=np.ndarray>
Attributes:
    detrend_orders:        [1]
    force_zero_mean:       True
    max_anomaly:           5.0
    method_anomaly:        detrend_fixed_baseline
    method_extreme:        hobday_extreme
    method_percentile:     approximate
    precision:             0.01
    preprocessing_steps:   ['Removed polynomial trend orders=[1]', 'Daily cli...
    threshold_percentile:  95
    window_days_hobday:    11

In [18]:
ds =xr.open_zarr('/home/b/b382616/scratch/mhws/hadgem/HadGEM3-LL_marEx_extremes_detrend_fixed_baseline_1982-2014.zarr')
ds

<xarray.Dataset> Size: 64GB
Dimensions:         (time: 12052, lat: 721, lon: 1440, dayofyear: 366)
Coordinates:
  * time            (time) datetime64[ns] 96kB 1982-01-01T12:00:00 ... 2014-1...
  * lat             (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon             (lon) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
Data variables:
    dat_anomaly     (time, lat, lon) float32 50GB dask.array<chunksize=(377, 46, 90), meta=np.ndarray>
    extreme_events  (time, lat, lon) bool 13GB dask.array<chunksize=(754, 46, 180), meta=np.ndarray>
    mask            (lat, lon) bool 1MB dask.array<chunksize=(361, 720), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 2GB dask.array<chunksize=(91, 180, 46), meta=np.ndarray>
Attributes:
    method_anomaly:        detrend_fixed_baseline
    method_extreme:        hobday_extreme
    threshold_percentile:  95
    preprocessing_steps:   ['Removed polynomial trend orders=[1]', 'Daily cli...
    detrend_orders:        [1]
    force_zero_mean:       True
    window_days_hobday:    11
    method_percentile:     approximate
    precision:             0.01
    max_anomaly:           5.0

<xarray.Dataset> Size: 64GB
Dimensions:         (time: 12052, lat: 721, lon: 1440, dayofyear: 366)
Coordinates:
  * time            (time) datetime64[ns] 96kB 1982-01-01T12:00:00 ... 2014-1...
  * lat             (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon             (lon) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
Data variables:
    dat_anomaly     (time, lat, lon) float32 50GB dask.array<chunksize=(377, 46, 90), meta=np.ndarray>
    extreme_events  (time, lat, lon) bool 13GB dask.array<chunksize=(754, 46, 180), meta=np.ndarray>
    mask            (lat, lon) bool 1MB dask.array<chunksize=(361, 720), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 2GB dask.array<chunksize=(91, 180, 46), meta=np.ndarray>
Attributes:
    method_anomaly:        detrend_fixed_baseline
    method_extreme:        hobday_extreme
    threshold_percentile:  95
    preprocessing_steps:   ['Removed polynomial trend orders=[1]', 'Daily cli...
    detrend_orders:        [1]
    force_zero_mean:       True
    window_days_hobday:    11
    method_percentile:     approximate
    precision:             0.01
    max_anomaly:           5.0

In [3]:
# For Zarr v3.0.0 and above, use LocalStore
from zarr.storage import LocalStore
import zarr
import xarray as xr

source_path = '/home/b/b382616/scratch/mhws/hadgem/HadGEM3-LL_marEx_extremes_detrend_fixed_baseline_1982-2014.zarr'
store = LocalStore(source_path)
root = zarr.open(store, mode='r')

data_vars = {}
coords = {}

# Map arrays to dimensions based on your zarr.json
for name, array in root.arrays():
    if name in ['lat', 'lon', 'time', 'dayofyear']:
        coords[name] = xr.DataArray(array[:], dims=[name])
    elif name in ['dat_anomaly', 'extreme_events']:
        data_vars[name] = xr.DataArray(array, dims=['time', 'lat', 'lon']).chunk({'time': 150})
    elif name == 'thresholds':
        data_vars[name] = xr.DataArray(array, dims=['lat', 'lon', 'dayofyear'])
    elif name == 'mask':
        data_vars[name] = xr.DataArray(array, dims=['lat', 'lon'])
    else:
        data_vars[name] = xr.DataArray(array)

ds = xr.Dataset(data_vars, coords=coords)
print(f"✓ Success! Dataset created with {list(ds.data_vars)}")

✓ Success! Dataset created with ['dat_anomaly', 'extreme_events', 'mask', 'thresholds']


In [6]:
import xarray as xr
import zarr
import dask.array as da
import numpy as np

def create_dataset_from_zarr(zarr_path):
    """Create xarray dataset from zarr store without _ARRAY_DIMENSIONS"""
    
    # Open zarr group
    zarr_group = zarr.open(zarr_path, mode='r')
    
    # First, load all 1D arrays as potential coordinates
    coords = {}
    for name in zarr_group.keys():
        arr = zarr_group[name]
        if arr.ndim == 1:
            coords[name] = arr[:]
    
    # Now create data variables with appropriate dimensions
    data_vars = {}
    
    for name in zarr_group.keys():
        if zarr_group[name].ndim == 1:
            continue  # Skip 1D arrays (already in coords)
            
        arr = zarr_group[name]
        
        # Try to guess dimensions based on shape matching
        dims = []
        for i, size in enumerate(arr.shape):
            # Check if this dimension size matches any coordinate
            matched = False
            for coord_name, coord_data in coords.items():
                if len(coord_data) == size:
                    dims.append(coord_name)
                    matched = True
                    break
            
            if not matched:
                # Check if it matches dayofyear (might be dayofyear dimension)
                if 'dayofyear' in coords and len(coords['dayofyear']) == size:
                    dims.append('dayofyear')
                else:
                    # Use generic dimension name
                    dims.append(f'dim_{i}')
        
        # Convert to dask array
        dask_arr = da.from_array(arr, chunks='auto')
        data_vars[name] = (dims, dask_arr)
    
    # Create dataset
    ds = xr.Dataset(data_vars)
    
    # Assign coordinates
    ds = ds.assign_coords({k: v for k, v in coords.items() if k in ['lat', 'lon', 'time']})
    
    # Handle dayofyear specially
    if 'dayofyear' in coords:
        if len(coords['dayofyear']) == len(ds.get('time', [])):
            ds = ds.assign_coords(dayofyear=('time', coords['dayofyear']))
        else:
            # Add as separate coordinate
            ds = ds.assign_coords(dayofyear=coords['dayofyear'])
    
    return ds

# Use the function
ds = create_dataset_from_zarr('/home/b/b382616/scratch/mhws/hadgem/HadGEM3-LL_marEx_extremes_detrend_fixed_baseline_1982-2014.zarr')
print(ds)

<xarray.Dataset> Size: 64GB
Dimensions:         (time: 12052, lat: 721, lon: 1440, dayofyear: 366)
Coordinates:
  * lat             (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon             (lon) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time            (time) float64 96kB 4.166e+09 4.166e+09 ... 5.207e+09
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
Data variables:
    dat_anomaly     (time, lat, lon) float32 50GB dask.array<chunksize=(754, 92, 180), meta=np.ndarray>
    extreme_events  (time, lat, lon) int8 13GB dask.array<chunksize=(1508, 92, 360), meta=np.ndarray>
    mask            (lat, lon) int8 1MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 2GB dask.array<chunksize=(273, 540, 138), meta=np.ndarray>
